In [1]:
import os
os.environ["AI2POT_PATH"] = "/data/home/liuhanyu/mycode/AI2Pot/"
os.environ["AI2POT_TEST_DATA_PATH"] = os.path.join(os.environ.get("AI2POT_PATH"),
                                                   "test",
                                                   "test_data",
                                                   "POSCARs")
poscar_path: str = os.path.join(os.environ.get("AI2POT_TEST_DATA_PATH"), "POSCAR")

from typing import List
import torch
from pymatgen.core import Structure
from ai2pot.models.mtp.linear_mtp import LinearMtp
from ai2pot.models.potential_train import LitLinearMtp
from ai2pot.utils.usepot import (
    MlffInput,
    MlffToEFLossInput,
    MlffToLossInput)

# 1. Inference

## 1.1. Load a model

In [2]:
lit_linear_mtp: LitLinearMtp = LitLinearMtp.load_from_checkpoint("/data/home/liuhanyu/mycode/AI2Pot/lightning_logs/linear_mtp/version_35/checkpoints/epoch=6-step=175.ckpt",
                                                                 map_location="cpu")
linear_mtp: LinearMtp = lit_linear_mtp.model

## 1.2. `MlffInput`

In [3]:
torch.manual_seed(42)

type_map: List[int] = [16, 34, 41, 75]
rcut: float = 6.0
umax_num_neighs: int = 100
pbc_xyz: List[bool] = [True, True, True]
sort: bool = True
dtype: torch._C.dtype = torch.float32
device: torch._C.device = torch.device("cpu")

linear_mtp: LinearMtp = LinearMtp(mtp_level=16,
                                  type_map=[16, 34, 41, 75],
                                  chebyshev_size=8,
                                  rmax=6.0,
                                  rmin=2.0,
                                  umax_num_neighs=100,
                                  fit_virial=False,
                                  zbl_rmax=2.0,
                                  zbl_rmin=1.0).to(dtype).to(device)

## 1.3. `predict_efv()`

In [4]:
structure: Structure = Structure.from_file(poscar_path)
mlff_input: MlffInput = MlffInput(type_map=type_map,
                                  rcut=rcut,
                                  umax_num_neighs=umax_num_neighs,
                                  pbc_xyz=pbc_xyz,
                                  sort=sort,
                                  dtype=dtype,
                                  device=device)

e, f, v = linear_mtp.predict_efv(*mlff_input.analyse_pymatgen(structure=structure))
print("1. Energy.shape = ", e.shape)
print("2. Force.shape = ", f.shape)
print("3. Virial.shape = ", v.shape)

1. Energy.shape =  torch.Size([1])
2. Force.shape =  torch.Size([1, 108, 3])
3. Virial.shape =  torch.Size([1, 9])


## 1.4. `predict_ef()`

In [5]:
structure: Structure = Structure.from_file(poscar_path)
mlff_input: MlffInput = MlffInput(type_map=type_map,
                                  rcut=rcut,
                                  umax_num_neighs=umax_num_neighs,
                                  pbc_xyz=pbc_xyz,
                                  sort=sort,
                                  dtype=dtype,
                                  device=device)

e, f = linear_mtp.predict_ef(*mlff_input.analyse_pymatgen(structure=structure))
print("1. Energy.shape = ", e.shape)
print("2. Force.shape = ", f.shape)

1. Energy.shape =  torch.Size([1])
2. Force.shape =  torch.Size([1, 108, 3])


## 1.5. `predict_loss()`

In [6]:
structure: Structure = Structure.from_file(poscar_path)
mlff_loss_input: MlffToLossInput = MlffToLossInput(type_map=type_map,
                                                   rcut=rcut,
                                                   umax_num_neighs=umax_num_neighs,
                                                   pbc_xyz=pbc_xyz,
                                                   sort=sort,
                                                   dtype=dtype,
                                                   device=device)

loss = linear_mtp.predict_loss(*mlff_loss_input.analyse_pymatgen(structure=structure,
                                                               e_weight=1.0,
                                                               f_weight=1.0,
                                                               v_weight=1.0))
print("1. loss = ", loss)

1. loss =  tensor([55.7920], grad_fn=<LinearMtpToLossFunction>>)


## 1.6. `predict_ef_loss()`

In [7]:
structure: Structure = Structure.from_file(poscar_path)
mlff_loss_input: MlffToEFLossInput = MlffToEFLossInput(type_map=type_map,
                                                   rcut=rcut,
                                                   umax_num_neighs=umax_num_neighs,
                                                   pbc_xyz=pbc_xyz,
                                                   sort=sort,
                                                   dtype=dtype,
                                                   device=device)

loss = linear_mtp.predict_ef_loss(*mlff_loss_input.analyse_pymatgen(structure=structure,
                                                               e_weight=1.0,
                                                               f_weight=1.0))
print("1. loss = ", loss)

1. loss =  tensor([9.9053], grad_fn=<LinearMtpToEFLossFunction>>)


## 1.7. `predict_descriptors()`

In [8]:
structure: Structure = Structure.from_file(poscar_path)
mlff_input: MlffInput = MlffInput(type_map=type_map,
                                  rcut=rcut,
                                  umax_num_neighs=umax_num_neighs,
                                  pbc_xyz=pbc_xyz,
                                  sort=sort,
                                  dtype=dtype,
                                  device=device)
descriptors = linear_mtp.predict_descriptors(*mlff_input.analyse_pymatgen(structure=structure))
print(descriptors.shape)

torch.Size([1, 108, 92])
